In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.0.3
numpy version: 1.24.4


In [2]:
# 如果自动寻找失败，可以把这里改成你的 CSV 完整路径。
# 例如：
# MANUAL_DATA_PATH = Path(r"D:/projects/ecommerce-user-analysis/data/raw/online_shoppers_intention.csv")
MANUAL_DATA_PATH = None

candidate_paths = []

if MANUAL_DATA_PATH is not None:
    candidate_paths.append(Path(MANUAL_DATA_PATH))

candidate_paths.extend([
    Path("../data/raw/online_shoppers_intention.csv"),
    Path("data/raw/online_shoppers_intention.csv"),
    Path("./online_shoppers_intention.csv"),
])

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    searched = "\n".join(f"- {p.resolve()}" for p in candidate_paths)
    raise FileNotFoundError(
        "没有找到 online_shoppers_intention.csv。\n"
        "请确认数据文件位置，或修改 MANUAL_DATA_PATH。\n"
        f"已尝试：\n{searched}"
    )

print("读取数据：", data_path.resolve())

读取数据： F:\Ecommerce-User-Analysis\data\raw\online_shoppers_intention.csv


In [3]:
df = pd.read_csv(data_path)

print("数据形状：", df.shape)
display(df.head())

数据形状： (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
required_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
    "Revenue",
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"缺少必要字段：{missing_columns}")

print("核心字段检查通过。")
print("总行数：", len(df))
print("缺失值总数：", int(df[required_columns].isna().sum().sum()))
print("完全重复行数：", int(df.duplicated().sum()))

核心字段检查通过。
总行数： 12330
缺失值总数： 0
完全重复行数： 125


In [ ]:
#统一布尔字段
def to_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f"}

    normalized = series.astype(str).str.strip().str.lower()

    unknown = set(normalized.dropna().unique()) - true_values - false_values
    if unknown:
        raise ValueError(f"字段 {series.name} 中存在无法识别的布尔值：{unknown}")

    return normalized.map(
        lambda x: True if x in true_values else False
    ).astype(bool)

df["Revenue"] = to_bool(df["Revenue"])
df["Weekend"] = to_bool(df["Weekend"])

df["is_purchase"] = df["Revenue"].astype(int)
df["is_weekend"] = df["Weekend"].astype(int)

df[["Revenue", "is_purchase", "Weekend", "is_weekend"]].head()

,Revenue,is_purchase,Weekend,is_weekend
0,False,0,False,0
1,False,0,False,0
2,False,0,False,0
3,False,0,False,0
4,False,0,True,1


In [6]:
#重新创建dashboard所需衍生字段
month_map = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "June": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}

df["month_num"] = df["Month"].map(month_map)

if df["month_num"].isna().any():
    unknown_months = df.loc[df["month_num"].isna(), "Month"].unique().tolist()
    raise ValueError(f"发现无法映射的月份：{unknown_months}")

df["product_duration_min"] = df["ProductRelated_Duration"] / 60

product_bins = [-np.inf, 5, 10, 20, 50, np.inf]
product_labels = ["0-5", "6-10", "11-20", "21-50", "50+"]

df["product_view_group"] = pd.cut(
    df["ProductRelated"],
    bins=product_bins,
    labels=product_labels
)

duration_bins = [-np.inf, 2, 5, 10, 20, np.inf]
duration_labels = ["0-2min", "2-5min", "5-10min", "10-20min", "20min+"]

df["duration_group"] = pd.cut(
    df["product_duration_min"],
    bins=duration_bins,
    labels=duration_labels,
    right=False
)

display(
    df[
        [
            "Month",
            "month_num",
            "ProductRelated",
            "product_view_group",
            "product_duration_min",
            "duration_group",
        ]
    ].head()
)

,Month,month_num,ProductRelated,product_view_group,product_duration_min,duration_group
0,Feb,2,1,0-5,0.000000,0-2min
1,Feb,2,2,0-5,1.066667,0-2min
2,Feb,2,1,0-5,0.000000,0-2min
3,Feb,2,2,0-5,0.044444,0-2min
4,Feb,2,10,6-10,10.458333,10-20min


In [7]:
#重新计算P75，并重新建立Session分层
p75 = df["ProductRelated"].quantile(0.75)

df["high_activity"] = df["ProductRelated"] >= p75

def segment_session(row):
    if row["high_activity"] and row["Revenue"]:
        return "High_Value"
    elif row["high_activity"] and not row["Revenue"]:
        return "High_Potential"
    elif (not row["high_activity"]) and row["Revenue"]:
        return "Low_Activity_Purchase"
    else:
        return "Low_Activity_NonPurchase"

df["session_segment"] = df.apply(segment_session, axis=1)

print("ProductRelated P75 =", p75)
display(df["session_segment"].value_counts().to_frame("session_cnt"))

ProductRelated P75 = 38.0


,session_cnt
session_segment,
Low_Activity_NonPurchase,8046
High_Potential,2376
Low_Activity_Purchase,1178
High_Value,730


In [8]:
#Dashboard总览KPI表
overview = pd.DataFrame({
    "total_sessions": [len(df)],
    "purchase_sessions": [int(df["is_purchase"].sum())],
    "conversion_rate": [df["is_purchase"].mean()],
    "high_potential_sessions": [
        int((df["session_segment"] == "High_Potential").sum())
    ],
    "avg_product_views": [df["ProductRelated"].mean()],
    "avg_product_duration_min": [df["product_duration_min"].mean()],
})

overview

,total_sessions,purchase_sessions,conversion_rate,high_potential_sessions,avg_product_views,avg_product_duration_min
0,12330,1908,0.154745,2376,31.731468,19.912437


In [9]:
#月度经营趋势表
monthly_summary = (
    df.groupby(["month_num", "Month"], as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
          avg_product_views=("ProductRelated", "mean"),
          avg_exit_rate=("ExitRates", "mean"),
      )
      .sort_values("month_num")
)

monthly_summary["conversion_rate"] = (
    monthly_summary["purchase_cnt"] / monthly_summary["session_cnt"]
)

monthly_summary

,month_num,Month,session_cnt,purchase_cnt,avg_product_views,avg_exit_rate,conversion_rate
0,2,Feb,184,3,11.184783,0.074148,0.016304
1,3,Mar,1907,192,19.808600,0.044600,0.100682
2,5,May,3364,365,26.487812,0.048850,0.108502
3,6,June,288,29,36.065972,0.058242,0.100694
4,7,Jul,432,66,36.407407,0.045330,0.152778
5,8,Aug,433,76,38.258661,0.037727,0.175520
6,9,Sep,448,86,33.104911,0.030320,0.191964
7,10,Oct,549,115,33.566485,0.029011,0.209472
8,11,Nov,2998,760,46.038692,0.038202,0.253502
9,12,Dec,1727,216,27.994789,0.041303,0.125072


In [10]:
#新老访客分析表
visitor_summary = (
    df.groupby("VisitorType", as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
          avg_product_views=("ProductRelated", "mean"),
          avg_product_duration_min=("product_duration_min", "mean"),
          avg_bounce_rate=("BounceRates", "mean"),
          avg_exit_rate=("ExitRates", "mean"),
      )
)

visitor_summary["conversion_rate"] = (
    visitor_summary["purchase_cnt"] / visitor_summary["session_cnt"]
)

visitor_summary = visitor_summary.sort_values(
    "conversion_rate",
    ascending=False
)

visitor_summary

,VisitorType,session_cnt,purchase_cnt,avg_product_views,avg_product_duration_min,avg_bounce_rate,avg_exit_rate,conversion_rate
0,New_Visitor,1694,422,18.054900,10.606556,0.005261,0.020681,0.249115
1,Other,85,16,12.470588,9.506748,0.038551,0.063349,0.188235
2,Returning_Visitor,10551,1470,34.082457,21.490358,0.024778,0.046505,0.139323


In [11]:
#商品浏览分析表
product_depth_summary = (
    df.groupby("product_view_group", observed=False, as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
      )
)

product_depth_summary["conversion_rate"] = (
    product_depth_summary["purchase_cnt"]
    / product_depth_summary["session_cnt"]
)

product_depth_summary

,product_view_group,session_cnt,purchase_cnt,conversion_rate
0,0-5,2369,102,0.043056
1,6-10,1804,185,0.102550
2,11-20,2560,392,0.153125
3,21-50,3445,685,0.198839
4,50+,2152,544,0.252788


In [12]:
#商品停留时间分析表
duration_summary = (
    df.groupby("duration_group", observed=False, as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
      )
)

duration_summary["conversion_rate"] = (
    duration_summary["purchase_cnt"]
    / duration_summary["session_cnt"]
)

duration_summary

,duration_group,session_cnt,purchase_cnt,conversion_rate
0,0-2min,2358,94,0.039864
1,2-5min,1808,133,0.073562
2,5-10min,2007,308,0.153463
3,10-20min,2377,482,0.202777
4,20min+,3780,891,0.235714


In [13]:
#TrafficType渠道分析表
traffic_summary = (
    df.groupby("TrafficType", as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
          avg_product_views=("ProductRelated", "mean"),
          avg_exit_rate=("ExitRates", "mean"),
      )
)

traffic_summary["conversion_rate"] = (
    traffic_summary["purchase_cnt"] / traffic_summary["session_cnt"]
)

traffic_summary = traffic_summary.sort_values(
    ["purchase_cnt", "conversion_rate"],
    ascending=[False, False]
)

traffic_summary.head(20)

,TrafficType,session_cnt,purchase_cnt,avg_product_views,avg_exit_rate,conversion_rate
1,2,3913,847,38.125224,0.026391,0.216458
0,1,2451,262,31.918401,0.055708,0.106895
2,3,2052,180,25.805556,0.057117,0.087719
3,4,1069,165,28.525725,0.036155,0.154350
7,8,343,95,26.122449,0.029639,0.276968
9,10,450,90,32.888889,0.037910,0.200000
4,5,260,56,17.884615,0.029679,0.215385
5,6,444,53,29.608108,0.045412,0.119369
19,20,198,50,20.146465,0.047548,0.252525
10,11,247,47,25.210526,0.043753,0.190283


In [14]:
#Session分层汇总表
segment_summary = (
    df.groupby("session_segment", as_index=False)
      .agg(
          session_cnt=("is_purchase", "size"),
          purchase_cnt=("is_purchase", "sum"),
          avg_product_views=("ProductRelated", "mean"),
          avg_product_duration_min=("product_duration_min", "mean"),
          avg_bounce_rate=("BounceRates", "mean"),
          avg_exit_rate=("ExitRates", "mean"),
          avg_page_value=("PageValues", "mean"),
      )
)

segment_summary["session_share"] = (
    segment_summary["session_cnt"] / len(df)
)

segment_summary["conversion_rate"] = (
    segment_summary["purchase_cnt"] / segment_summary["session_cnt"]
)

segment_summary

,session_segment,session_cnt,purchase_cnt,avg_product_views,avg_product_duration_min,avg_bounce_rate,avg_exit_rate,avg_page_value,session_share,conversion_rate
0,High_Potential,2376,0,80.864478,49.032701,0.007614,0.021814,4.182961,0.192701,0.0
1,High_Value,730,730,96.050685,59.883390,0.004543,0.015758,20.525807,0.059205,1.0
2,Low_Activity_NonPurchase,8046,0,13.314691,8.619834,0.030545,0.054927,1.324277,0.652555,0.0
3,Low_Activity_Purchase,1178,1178,18.563667,13.538702,0.005473,0.021909,31.440460,0.095539,1.0


In [15]:
#高潜Session明细表
high_potential_detail = df.loc[
    df["session_segment"] == "High_Potential",
    [
        "Month",
        "month_num",
        "VisitorType",
        "TrafficType",
        "Weekend",
        "ProductRelated",
        "ProductRelated_Duration",
        "product_duration_min",
        "BounceRates",
        "ExitRates",
        "PageValues",
        "SpecialDay",
        "session_segment",
    ],
].copy()

high_potential_detail.insert(
    0,
    "session_row_id",
    high_potential_detail.index.astype(int)
)

print("High Potential Session 数：", len(high_potential_detail))
display(high_potential_detail.head())

High Potential Session 数： 2376


,session_row_id,Month,month_num,VisitorType,TrafficType,Weekend,ProductRelated,ProductRelated_Duration,product_duration_min,BounceRates,ExitRates,PageValues,SpecialDay,session_segment
29,29,Feb,2,Returning_Visitor,1,False,45,1582.750000,26.379167,0.043478,0.050821,54.179764,0.4,High_Potential
35,35,Feb,2,Returning_Visitor,1,False,52,2086.242857,34.770714,0.015385,0.020353,0.000000,0.0,High_Potential
40,40,Feb,2,Returning_Visitor,4,False,46,4084.393939,68.073232,0.000000,0.001795,0.000000,0.0,High_Potential
62,62,Feb,2,Returning_Visitor,2,False,42,1553.583333,25.893056,0.009000,0.019667,38.308493,0.0,High_Potential
66,66,Feb,2,Returning_Visitor,3,False,90,6951.972222,115.866204,0.002151,0.015013,0.000000,0.0,High_Potential


In [16]:
#导出前数据质量校验
assert len(df) > 0, "数据为空"
assert overview.loc[0, "total_sessions"] == len(df)
assert overview.loc[0, "purchase_sessions"] == int(df["is_purchase"].sum())
assert 0 <= overview.loc[0, "conversion_rate"] <= 1
assert monthly_summary["session_cnt"].sum() == len(df)
assert visitor_summary["session_cnt"].sum() == len(df)
assert product_depth_summary["session_cnt"].sum() == len(df)
assert duration_summary["session_cnt"].sum() == len(df)
assert traffic_summary["session_cnt"].sum() == len(df)
assert segment_summary["session_cnt"].sum() == len(df)

print("Dashboard 数据质量校验全部通过。")

# 如果使用的是课程原始 UCI 数据，通常还可以额外看到：
print("Total Sessions =", len(df))
print("Purchase Sessions =", int(df["is_purchase"].sum()))
print("CVR =", f'{df["is_purchase"].mean():.2%}')

Dashboard 数据质量校验全部通过。
Total Sessions = 12330
Purchase Sessions = 1908
CVR = 15.47%


In [ ]:
#导出Power BI 数据文件
output_dir = Path("../data/processed/powerbi")
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "01_overview.csv": overview,
    "02_monthly_summary.csv": monthly_summary,
    "03_visitor_summary.csv": visitor_summary,
    "04_product_depth_summary.csv": product_depth_summary,
    "05_duration_summary.csv": duration_summary,
    "06_traffic_summary.csv": traffic_summary,
    "07_segment_summary.csv": segment_summary,
    "08_high_potential_detail.csv": high_potential_detail,
}

for filename, table in exports.items():
    path = output_dir / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"已导出: {path.resolve()}  |  rows={len(table)}")

已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\01_overview.csv  |  rows=1
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\02_monthly_summary.csv  |  rows=10
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\03_visitor_summary.csv  |  rows=3
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\04_product_depth_summary.csv  |  rows=5
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\05_duration_summary.csv  |  rows=5
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\06_traffic_summary.csv  |  rows=20
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\07_segment_summary.csv  |  rows=4
已导出: F:\Ecommerce-User-Analysis\data\processed\powerbi\08_high_potential_detail.csv  |  rows=2376


# 导出 Power BI Session 明细表

为了支持Power BI中的VisitorType、Month、TrafficType、
Session Segment等交互式筛选，这里额外导出处理后的Session级明细数据。

数据粒度：

一行 = 一个Session

注意：不是独立用户明细表，因为原始数据不存在user_id。

In [18]:
powerbi_detail_columns = [
    "Month",
    "month_num",
    "VisitorType",
    "TrafficType",
    "Weekend",
    "is_weekend",
    "ProductRelated",
    "ProductRelated_Duration",
    "product_duration_min",
    "product_view_group",
    "duration_group",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Revenue",
    "is_purchase",
    "high_activity",
    "session_segment",
]

session_detail = df[powerbi_detail_columns].copy()

session_detail.insert(
    0,
    "session_id",
    range(1, len(session_detail) + 1)
)

print("Session明细行数：", len(session_detail))

display(session_detail.head())

Session明细行数： 12330


,session_id,Month,month_num,VisitorType,TrafficType,Weekend,is_weekend,ProductRelated,ProductRelated_Duration,product_duration_min,product_view_group,duration_group,BounceRates,ExitRates,PageValues,SpecialDay,Revenue,is_purchase,high_activity,session_segment
0,1,Feb,2,Returning_Visitor,1,False,0,1,0.000000,0.000000,0-5,0-2min,0.20,0.20,0.0,0.0,False,0,False,Low_Activity_NonPurchase
1,2,Feb,2,Returning_Visitor,2,False,0,2,64.000000,1.066667,0-5,0-2min,0.00,0.10,0.0,0.0,False,0,False,Low_Activity_NonPurchase
2,3,Feb,2,Returning_Visitor,3,False,0,1,0.000000,0.000000,0-5,0-2min,0.20,0.20,0.0,0.0,False,0,False,Low_Activity_NonPurchase
3,4,Feb,2,Returning_Visitor,4,False,0,2,2.666667,0.044444,0-5,0-2min,0.05,0.14,0.0,0.0,False,0,False,Low_Activity_NonPurchase
4,5,Feb,2,Returning_Visitor,4,True,1,10,627.500000,10.458333,6-10,10-20min,0.02,0.05,0.0,0.0,False,0,False,Low_Activity_NonPurchase


In [19]:
#再做一次数据质量检查
assert len(session_detail) == len(df)

assert session_detail["session_id"].nunique() == len(df)

assert session_detail["is_purchase"].sum() == df["is_purchase"].sum()

assert (
    session_detail["session_segment"]
    .eq("High_Potential")
    .sum()
    ==
    (df["session_segment"] == "High_Potential").sum()
)

print("Session明细表数据质量检查通过。")

Session明细表数据质量检查通过。


In [20]:
#导出CSV
session_detail_path = output_dir / "09_session_detail.csv"

session_detail.to_csv(
    session_detail_path,
    index=False,
    encoding="utf-8-sig"
)

print("已导出：")
print(session_detail_path.resolve())

已导出：
F:\Ecommerce-User-Analysis\data\processed\powerbi\09_session_detail.csv
